In [3]:

import os
import polars as pl 
from dotenv import load_dotenv
from datasets import load_dataset
from huggingface_hub import HfFileSystem

load_dotenv()

/Users/gsmith/Documents/Projects/Huggingface/hf-usa-spending/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
HF_TOKEN=os.getenv("HF_TOKEN")
HF_NAMESPACE=os.getenv("HF_NAMESPACE")
HF_BUCKET_NAME=os.getenv("HF_BUCKET_NAME")

In [5]:
!hf buckets list

ID                       PRIVATE       SIZE TOTAL_FILES CREATED_AT
------------------------ ------- ---------- ----------- ----------
ggsmith/usa-spending-raw         2734980740           8 2026-04-04


In [6]:
bucket_id = f"{HF_NAMESPACE}/{HF_BUCKET_NAME}"

fs = HfFileSystem()
# Get all parquet files in the bucket folder
files = fs.glob(f"hf://buckets/{bucket_id}/parquet/*.parquet")

# We open them individually because Polars scan_parquet doesn't 
# support a list of fsspec file objects directly as well as it does local paths.
# For EDA on multiple bucket files, it's often easiest to load them into a list:

# dfs = []
# for f_path in files[0]:
#     with fs.open(f"hf://{f_path}", mode="rb") as f:
#         dfs.append(pl.read_parquet(f))

# # Combine for analysis
# df = pl.concat(dfs)
# print(df.describe())

In [10]:
data = []
with fs.open(f"hf://{files[0]}", mode="rb") as f:
    data.append(pl.read_parquet(f))
    

In [18]:
data[0].select(
        [
            pl.col("award_id_piid").alias("Award ID"),
            pl.col("recipient_name").alias("Recipient Name"),
            pl.col("recipient_duns").alias("Recipient DUNS Number"),
            pl.col("period_of_performance_start_date").alias("Start Date"),
            pl.col("period_of_performance_current_end_date").alias("End Date"),
            pl.col("transaction_description").alias("Description"),
            pl.col("current_total_value_of_award").alias("Award Amount"),
            pl.col("awarding_agency_name").alias("Awarding Agency"),
            pl.col("awarding_sub_agency_name").alias("Awarding Sub Agency"),
            pl.col("award_type").alias("Contract Award Type"),
            pl.col("award_or_idv_flag").alias("Award Type"),
            pl.col("funding_agency_name").alias("Funding Agency"),
            pl.col("funding_sub_agency_name").alias("Funding Sub Agency"),
            pl.col("naics_code").alias("NAICS"),
            pl.col("product_or_service_code").alias("PSC"),
        ]
    )

Award ID,Recipient Name,Recipient DUNS Number,Start Date,End Date,Description,Award Amount,Awarding Agency,Awarding Sub Agency,Contract Award Type,Award Type,Funding Agency,Funding Sub Agency,NAICS,PSC
str,str,str,date,date,str,f64,str,str,str,str,str,str,str,str
"""W81XWH19C0137""","""CHENEGA HEALTHCARE SERVICES LL…",null,2019-07-11,2024-10-31,"""THIS REQUIREMENT IS FOR JANITO…",7.6702e6,"""Department of Defense""","""Defense Health Agency""","""DEFINITIVE CONTRACT""","""AWARD""","""Department of Defense""","""Defense Health Agency""","""561720""","""AN91"""
"""W90VN624PA023""","""SNB CO., LTD""",null,2024-09-18,2024-10-18,"""THIS IS FOR THE PURCHASE OF A …",16371.78,"""Department of Defense""","""Department of the Army""","""PURCHASE ORDER""","""AWARD""","""Department of Defense""","""Department of the Air Force""","""339992""","""5835"""
"""1145PC23F0487""","""MARTIN, CYNTHIA L""",null,2023-08-07,2023-08-18,"""BACK-TO-BACK LPI WORKSHOPS FOR…",8800.0,"""Peace Corps""","""Peace Corps""","""BPA CALL""","""AWARD""","""Peace Corps""","""Peace Corps""","""541930""","""R608"""
"""1145PC24F0353""","""MORE PREPARED LLC""",null,2024-11-01,2024-12-15,"""MEDICAL KITS BPA""",241500.0,"""Peace Corps""","""Peace Corps""","""BPA CALL""","""AWARD""","""Peace Corps""","""Peace Corps""","""339113""","""6515"""
"""1131PL22FIQ11175""","""WEBSTER GROUP INC""",null,2022-08-24,2025-02-23,"""SOUTH AFRICA CLEAN ENERGY AND …",974746.0,"""United States Trade and Develo…","""United States Trade and Develo…","""DELIVERY ORDER""","""AWARD""","""United States Trade and Develo…","""United States Trade and Develo…","""541611""","""R499"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""47QSCC24F41GV""","""W.W. GRAINGER, INC.""",null,2024-03-28,2024-04-04,"""TONER CARTRIDGE MAGENTA HEWCF3…",1240.94,"""General Services Administratio…","""Federal Acquisition Service""","""BPA CALL""","""AWARD""","""General Services Administratio…","""Federal Acquisition Service""","""444110""","""5340"""
"""47QSCC24F41JM""","""M-80 SYSTEMS, INC.""",null,2024-03-28,2024-03-31,"""PLATE, PAPER (ROUND, 3-COMPART…",2373.8,"""General Services Administratio…","""Federal Acquisition Service""","""DELIVERY ORDER""","""AWARD""","""General Services Administratio…","""Federal Acquisition Service""","""493190""","""R499"""
"""47QSCC24F41VB""","""W.W. GRAINGER, INC.""",null,2024-03-29,2024-04-05,"""PROPRESS 90 ELBOW, 2"" X 2""""",203.6,"""General Services Administratio…","""Federal Acquisition Service""","""BPA CALL""","""AWARD""","""General Services Administratio…","""Federal Acquisition Service""","""444110""","""5120"""


In [21]:
data[0].filter(pl.col("recipient_duns").is_not_null()).select("recipient_duns")

recipient_duns
str
